In [1]:
"""
backward-model-corrected.py
============================
Complete, self-contained script equivalent to backward-model.ipynb with
all GPU-optimisation and correctness fixes applied.

Run on Kaggle with GPU enabled. Sections are labelled by their original
notebook-cell numbers for easy cross-referencing.

Changes vs original notebook:
  - BATCH_SIZE_STARS  5  â†’ 10
  - NUM_BURNIN       400 â†’ 1000,  NUM_ADAPT 320 â†’ 800
  - Per-star step-size adaptation  (shape 1,B,23  not  23,)
  - tf.repeat broadcast  (explicit starâ†”chain alignment)
  - .numpy() calls removed from graph-traced code
  - Single XLA compilation via tf.Variable
  - Last-batch padding + reduced checkpoint frequency
"""


import os
import glob
import h5py
import numpy as np
import tensorflow as tf
from tensorflow.keras.saving import register_keras_serializable
from tensorflow.keras import layers, models, Input, callbacks, regularizers
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
import matplotlib.pyplot as plt
from scipy.stats import binned_statistic
import seaborn as sns
from scipy.stats import spearmanr
from matplotlib.lines import Line2D

#-----------------------------------------------------------------
# Configs
#-----------------------------------------------------------------

class Config:
    # --- Paths ---
    H5_PATH = "/kaggle/input/datasets/aneeshshastri/aspcapstar-dr17-150kstars/apogee_dr17_parallel.h5" 
    TFREC_DIR = "/kaggle/working/tfrecords"
    STATS_PATH = "/kaggle/working/dataset_stats.npz"
    start=140_000
    end=149_500
    # --- System ---
    TESTING_MODE = True
    TEST_LIMIT = None
    NUM_SHARDS = 16 
    TRAIN_INTEGRATED=True
    TRAIN_BASE_MODEL=True
    RANDOM_SEED=42
    # --- Model Hyperparameters ---
    BATCH_SIZE = 64     
    LEARNING_RATE = 1e-3  
    EPOCHS = 50
    LATENT_DIM = 268
    OUTPUT_LENGTH = 8575
    # --loss related---  
    IVAR_SCALE = 1000.0   
    CLIP_NORM = 1.0     
    BADPIX_CUTOFF=1e-3  
    #----predictor-labels--------
    #CAUTION: LITERALLY EVERYTHING IS IN THE SAME ORDER AS THESE LABELS. DO NOT TOUCH THE ORDER OF THESE LABELS
    SELECTED_LABELS = [
        # 1. Core
        'TEFF', 'LOGG', 'M_H', 'VMICRO', 'VMACRO', 'VSINI',
        # 2. CNO
        'C_FE', 'N_FE', 'O_FE',
        #3. metals
        'FE_H',
        'MG_FE', 'SI_FE', 'CA_FE', 'TI_FE', 'S_FE',
        'AL_FE', 'MN_FE', 'NI_FE', 'CR_FE','K_FE',
        'NA_FE','V_FE','CO_FE'
    ]
    ABUNDANCE_INDICES =[i for i, label in enumerate(SELECTED_LABELS) if '_FE' in label]
    FE_H_INDEX = SELECTED_LABELS.index('FE_H')
    N_LABELS = len(SELECTED_LABELS) + 4
    ERRORS=[f'{label}_ERR' for label in SELECTED_LABELS ]
    #GRAPHING:
    WAVELENGTH_START = 1514
    WAVELENGTH_END = 1694 


config = Config()
np.random.seed(config.RANDOM_SEED)
tf.random.set_seed(config.RANDOM_SEED)
os.makedirs(config.TFREC_DIR, exist_ok=True)

2026-04-02 18:20:41.910527: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775154042.097397      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775154042.153052      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775154042.589381      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775154042.589418      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775154042.589421      55 computation_placer.cc:177] computation placer alr

In [2]:
from sklearn.decomposition import PCA


def get_clean_imputed_data(h5_path, selected_labels, limit=None):
    
    print("Read data for imputation")
    
    with h5py.File(h5_path, 'r') as f:
        get_col = lambda k: f['metadata'][k]
        keys = f['metadata'].dtype.names
        print(np.mean((f['metadata']['SNR'])[config.start:]))
        raw_values = np.stack([get_col(p) for p in selected_labels], axis=1)
        bad_mask = np.zeros_like(raw_values, dtype=bool)
        
        for i, label in enumerate(selected_labels):
            flag_name = f"{label}_FLAG"
            if flag_name in keys:
                flg = get_col(flag_name)
                #Handle Void/Structured Types safely
                if flg.dtype.names: flg = flg[flg.dtype.names[0]]
                if flg.dtype.kind == 'V': flg = flg.view('<i4')
                is_bad = (flg.astype(int) != 0)
            elif label in ['TEFF', 'LOGG', 'VMICRO', 'VMACRO', 'VSINI']:
                is_bad = (raw_values[:, i] < -5000)
            else:
                is_bad = np.zeros_like(raw_values[:, i], dtype=bool)
            bad_mask[:, i] = is_bad

    if limit:
        print(f"Truncating to first {limit} stars.")
        raw_values = raw_values[:limit]
        bad_mask = bad_mask[:limit]

    print(f"Imputing Labels for {len(raw_values)} stars...")
    vals_to_impute = raw_values.copy()
    vals_to_impute[bad_mask] = np.nan
    
    imputer = IterativeImputer(estimator=BayesianRidge(), max_iter=10, initial_strategy='median')
    clean_labels = imputer.fit_transform(vals_to_impute)
    print("Converting [X/Fe] to [X/H]...")
    
    # Get the Iron Abundance column
    fe_h_col = clean_labels[:, config.FE_H_INDEX : config.FE_H_INDEX+1] 
    
    # Loop through all metal columns and add iron
    for idx in config.ABUNDANCE_INDICES:
        clean_labels[:, idx] = clean_labels[:, idx] + fe_h_col[:, 0]
        
    try:
        #calculate Inverse temperature to help with spectral lines
        teff_idx = selected_labels.index('TEFF')
        teff_vals = clean_labels[:, teff_idx]
        
        inv_teff = 5040.0 / (teff_vals + 1e-6)
        inv_teff = inv_teff.reshape(-1, 1)
        
        # Stack it onto the end of the array
        clean_labels = np.hstack([clean_labels, inv_teff])
        
    except ValueError:
        print("TEFF not found in labels")
         
    vmacro_idx = selected_labels.index('VMACRO')
    vmacro_vals = clean_labels[:, vmacro_idx]
       
    vsini_idx = selected_labels.index('VSINI')
    vsini_vals = clean_labels[:, vsini_idx]
    quadrature=np.sqrt(np.square(vmacro_vals)+np.square(vsini_vals))
    quadrature = quadrature.reshape(-1, 1)
    clean_labels = np.hstack([clean_labels, quadrature])
    
    try:
        C_idx = selected_labels.index('C_FE')
        C_vals = clean_labels[:, C_idx]
       
        O_idx = selected_labels.index('O_FE')
        O_vals = clean_labels[:, O_idx]
        C_O_diff=C_vals-O_vals
        C_O_diff=C_O_diff.reshape(-1,1)
        clean_labels = np.hstack([clean_labels, C_O_diff])
        
    except ValueError:
        print("C_FE or O_FE not found in labels")
    
    try:
        G_idx = selected_labels.index('LOGG')
        G_vals = clean_labels[:, G_idx]
       
        M_idx = selected_labels.index('M_H')
        M_vals = clean_labels[:, M_idx]
        LOGPE=0.5*G_vals+0.5*M_vals
        LOGPE=LOGPE.reshape(-1,1)
        clean_labels = np.hstack([clean_labels, LOGPE])
        
    except ValueError:
        print("M_H or LOGG not found in labels")

    print(f"Transformation Complete. Final Input Shape: {clean_labels.shape}")
    
    return clean_labels

def get_err(h5_path, selected_labels, limit=None):
    
    print("Read data for imputation")
    
    with h5py.File(h5_path, 'r') as f:
        get_col = lambda k: f['metadata'][k]
        keys = f['metadata'].dtype.names
        print(np.mean(f['metadata']['SNR']))
        raw_values = np.stack([get_col(p) if p not in ['VMACRO_ERR','VMICRO_ERR','VSINI_ERR'] else np.zeros_like(get_col('TEFF_ERR'))  for p in selected_labels 
                              ], axis=1)
        
        bad_mask = np.zeros_like(raw_values, dtype=bool)
        for i, label in enumerate(selected_labels):
            flag_name = f"{label}_FLAG"
            if flag_name in keys:
                flg = get_col(flag_name)
                #Handle Void/Structured Types safely
                if flg.dtype.names: flg = flg[flg.dtype.names[0]]
                if flg.dtype.kind == 'V': flg = flg.view('<i4')
                is_bad = (flg.astype(int) != 0)
            elif label in ['TEFF', 'LOGG', 'VMICRO', 'VMACRO', 'VSINI']:
                is_bad = (raw_values[:, i] < -5000)
            else:
                is_bad = np.zeros_like(raw_values[:, i], dtype=bool)
            bad_mask[:, i] = is_bad

        if limit:
            print(f"Truncating to first {limit} stars.")
            raw_values = raw_values[:limit]
            bad_mask = bad_mask[:limit]

    vals_to_impute = raw_values.copy()
    vals_to_impute[bad_mask] = np.nan
    bad_error_values = np.nanpercentile(vals_to_impute, 95, axis=0)

    nan_indices = np.isnan(vals_to_impute)
    clean_labels = np.where(nan_indices, bad_error_values, vals_to_impute)
         
    return clean_labels

In [3]:
@register_keras_serializable()
def chi2_estimate(y_true,y_pred):
    y_true=tf.cast(y_true,tf.float32)
    y_pred=tf.cast(y_pred,tf.float32)
    if len(y_pred.shape) == 2:
        y_pred = tf.expand_dims(y_pred, -1)
    real_flux = y_true[:, :, 0:1]
    ivar = y_true[:, :, 1:2]
    valid_mask = tf.cast(real_flux > config.BADPIX_CUTOFF, tf.float32)    
    safe_flux = tf.where(valid_mask == 1.0, real_flux, y_pred)
    weight=1.0/(1/ivar+1e-5)
    weight=tf.where(ivar>0,weight,0.0)
    wmse_term = tf.square(safe_flux - y_pred) * weight * valid_mask
    n_valid_pixels = tf.reduce_sum(valid_mask, 1)
    n_params = len(config.SELECTED_LABELS)
    dof = n_valid_pixels - n_params
    return tf.reduce_sum(wmse_term,1)/dof
    


@register_keras_serializable()
def spl_loss(y_true, y_pred):
    if len(y_pred.shape) == 2:
        y_pred = tf.expand_dims(y_pred, -1)
    real_flux = y_true[:, :, 0:1]
    ivar = y_true[:, :, 1:2]
    valid_mask = tf.cast(real_flux > config.BADPIX_CUTOFF, tf.float32)    
    safe_flux = tf.where(valid_mask == 1.0, real_flux, y_pred)
    weight=1e-5/(1/ivar+1e-5)
    weight=tf.where(ivar>0,weight,0.0)
    wmse_term = tf.square(safe_flux - y_pred) * weight * valid_mask
    
    loss = tf.where(tf.math.is_finite(wmse_term), wmse_term, tf.zeros_like(wmse_term))
    
    return loss

In [4]:
@tf.keras.utils.register_keras_serializable()
class ColumnSelector(layers.Layer):
    """
    A robust layer that selects specific columns from the input.
    Replaces: Lambda(lambda x: tf.gather(x, indices, axis=1))
    """
    def __init__(self, indices, **kwargs):
        super().__init__(**kwargs)
        self.indices = list(indices) 

    def call(self, inputs):
        return tf.gather(inputs, self.indices, axis=1)

    def get_config(self):
        config = super().get_config()
        config.update({'indices': self.indices})
        return config

@tf.keras.utils.register_keras_serializable()
class BeerLambertLaw(layers.Layer):
    """
    Applies the Beer-Lambert law: Flux = exp(-Tau).
    Replaces: Lambda(lambda t: tf.math.exp(-t))
    """
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self,k ,tau):
        return k*tf.math.exp(-tau)

@tf.keras.utils.register_keras_serializable()
class GetAbundances(layers.Layer):
    """
     Returns the abundance column for each element 
    """
    def __init__(self,col_id, **kwargs):
        super().__init__(**kwargs)
        self.index=col_id

    def call(self, inputs):
        return inputs[:, self.index:self.index+1]

In [5]:
@tf.keras.utils.register_keras_serializable()
class GetAbundances(layers.Layer):
    """Slices a single column from the input."""
    def __init__(self, col_id, **kwargs):
        super().__init__(**kwargs)
        self.index = col_id

    def call(self, inputs):
        return inputs[:, self.index : self.index + 1]
        
@tf.keras.utils.register_keras_serializable()
class SparseProjector(tf.keras.layers.Layer):
    def __init__(self, active_indices, weights, total_pixels, label_name="unknown", **kwargs):
        # Override name before passing to super to avoid conflict in from_config
        kwargs['name'] = f"Sparse_Projector_{label_name}"
        super().__init__(**kwargs)
        self.total_pixels = total_pixels
        self.label_name = label_name
        self.active_indices = tf.constant(active_indices, dtype=tf.int32)
        self.weights_tensor = tf.constant(weights, dtype=tf.float32)

    def call(self, local_tau):
    # Always compute in float32 — do NOT cast back to input_dtype.
    # Casting back to float16 severs the gradient tape for NUTS/HMC.
        local_tau_32 = tf.cast(local_tau, tf.float32)
        weighted = local_tau_32 * self.weights_tensor[None, :]
    
        batch_size = tf.shape(local_tau)[0]
        n_active = tf.shape(self.active_indices)[0]
    
        batch_ids = tf.repeat(tf.range(batch_size), n_active)
        pixel_ids = tf.tile(self.active_indices, tf.expand_dims(batch_size, axis=0))
    
        scatter_idx = tf.stack([batch_ids, pixel_ids], axis=-1)
        flat_weighted = tf.reshape(weighted, [-1])
    
        full = tf.scatter_nd(
            scatter_idx,
            flat_weighted,
            shape=tf.stack([batch_size, tf.constant(self.total_pixels, dtype=tf.int32)])
        )
        return full  # float32 always — no cast back

    def compute_output_shape(self, input_shape):
        return (input_shape[0], self.total_pixels)

    def get_config(self):
        config = super().get_config()
        config.update({
            'active_indices': self.active_indices.numpy().tolist(),
            'weights': self.weights_tensor.numpy().tolist(),
            'total_pixels': self.total_pixels,
            'label_name': self.label_name,
        })
        return config

    @classmethod
    def from_config(cls, config):
        config['active_indices'] = np.array(config['active_indices'], dtype=np.int32)
        config['weights'] = np.array(config['weights'], dtype=np.float32)
        config.pop('name', None)  # remove saved name, __init__ constructs it from label_name
        return cls(**config)

In [6]:
N_LABELS=23
N_PIXELS=8575
@register_keras_serializable()
class HeteroscedasticCNNPredictor(tf.keras.Model):
    """
    1D CNN that predicts stellar labels + per-label log-variance
    from an 8575-pixel APOGEE spectrum.

    Forward pass returns (mean, log_var) both shaped (batch, 23).
    """

    def __init__(self, n_labels=N_LABELS, **kwargs):
        super().__init__(**kwargs)
        self.n_labels = n_labels

        # ── convolutional backbone ──
        self.reshape_in = layers.Reshape((N_PIXELS, 1))

        self.conv1 = layers.Conv1D(32,  kernel_size=16, strides=4,
                                   activation='relu', padding='same')
        self.bn1   = layers.BatchNormalization()

        self.conv2 = layers.Conv1D(64,  kernel_size=8, strides=4,
                                   activation='relu', padding='same')
        self.bn2   = layers.BatchNormalization()

        self.conv3 = layers.Conv1D(128, kernel_size=4, strides=2,
                                   activation='relu', padding='same')
        self.bn3   = layers.BatchNormalization()

        self.conv4 = layers.Conv1D(256, kernel_size=4, strides=2,
                                   activation='relu', padding='same')
        self.bn4   = layers.BatchNormalization()

        self.gap   = layers.GlobalAveragePooling1D()

        # ── dense head ──
        self.dense1  = layers.Dense(512, activation='relu')
        self.drop1   = layers.Dropout(0.3)
        self.dense2  = layers.Dense(256, activation='relu')
        self.drop2   = layers.Dropout(0.2)

        # ── output heads ──
        self.mean_head    = layers.Dense(n_labels, name='label_mean')
        self.log_var_head = layers.Dense(n_labels, name='label_log_var')

    def call(self, x, training=False):
        # x: (batch, 8575)
        h = self.reshape_in(x)

        h = self.bn1(self.conv1(h), training=training)
        h = self.bn2(self.conv2(h), training=training)
        h = self.bn3(self.conv3(h), training=training)
        h = self.bn4(self.conv4(h), training=training)
        h = self.gap(h)

        h = self.drop1(self.dense1(h), training=training)
        h = self.drop2(self.dense2(h), training=training)

        mu      = self.mean_head(h)             # (batch, 23)
        log_var = self.log_var_head(h)           # (batch, 23)
        return mu, log_var

    def get_config(self):
        cfg = super().get_config()
        cfg['n_labels'] = self.n_labels
        return cfg


In [7]:
@register_keras_serializable()
class TrainableCNNPredictor(HeteroscedasticCNNPredictor):
    """
    Thin subclass that adds train_step / test_step so Keras' .fit()
    works with our tuple-output (mu, log_var) model — no wrapper needed.
    The model itself is what gets checkpointed and saved.
    """

    def train_step(self, data):
        x, y = data
        with tf.GradientTape() as tape:
            mu, raw_log_var = self(x, training=True)
            loss = _heteroscedastic_nll(y, mu, raw_log_var)
        grads = tape.gradient(loss, self.trainable_variables)
        # Clip gradients to prevent spikes from variance-head feedback
        grads, _ = tf.clip_by_global_norm(grads, 1.0)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        # Track MSE separately so we can see mean quality independent of variance
        mse = tf.reduce_mean(tf.square(y - mu))
        return {"loss": loss, "mse": mse}

    def test_step(self, data):
        x, y = data
        mu, raw_log_var = self(x, training=False)
        loss = _heteroscedastic_nll(y, mu, raw_log_var)
        mse  = tf.reduce_mean(tf.square(y - mu))
        return {"loss": loss, "mse": mse}


In [8]:
n_pixels = 8575
h5_path="/kaggle/input/datasets/aneeshshastri/aspcapstar-dr17-150kstars/apogee_dr17_parallel.h5"
stats_path="/kaggle/input/models/aneeshshastri/stargen-comparision/tensorflow2/default/14/dataset_stats_120k.npz"
model_path="/kaggle/input/models/aneeshshastri/stargen-comparision/tensorflow2/default/14/APOGEE_SGEN.keras"
jacobian_mask_path="/kaggle/input/datasets/aneeshshastri/element-masks/apogee_mask.npy"
model_path_P="/kaggle/input/models/aneeshshastri/stargen-comparision/tensorflow2/default/9/ThePayne.keras"
model_path_EP="/kaggle/input/models/aneeshshastri/stargen-comparision/tensorflow2/default/9/CNN_model.keras"
model_path_CNN="/kaggle/input/models/aneeshshastri/stargen-comparision/tensorflow2/default/14/SPECTROGRAM_GENERATOR.keras"
model_path_RP="/kaggle/input/models/aneeshshastri/stargen-comparision/tensorflow2/default/14/RegPayne.keras"

# --- CNN Warm-Start Model Paths ---
cnn_model_path = "/kaggle/input/models/aneeshshastri/backward-warmstart/tensorflow2/default/1/cnn_label_predictor_inner (1).keras"
cnn_stats_path = "/kaggle/input/models/aneeshshastri/backward-warmstart/tensorflow2/default/1/cnn_label_stats.npz"

with h5py.File(h5_path, 'r') as f:
    real_data = f['flux'][config.start:config.end]
    real_data_ivar = f['ivar'][config.start:config.end]

with np.load(stats_path) as data:
    MEAN_TENSOR= data['mean'].astype(np.float32)
    STD_TENSOR = data['std'].astype(np.float32)
    labels=get_clean_imputed_data(h5_path, config.SELECTED_LABELS)
    label_err=get_err(h5_path, config.ERRORS)[config.start:config.end]
    print(labels.shape,MEAN_TENSOR.shape,STD_TENSOR.shape)
    labels=((labels-MEAN_TENSOR)/(STD_TENSOR))[config.start:config.end]



model=tf.keras.models.load_model(model_path)





cnn_predictor = tf.keras.models.load_model(cnn_model_path)

with np.load(cnn_stats_path) as cnn_stats:
    CNN_LABEL_MEAN = cnn_stats['mean'].astype(np.float32)   # (23,)
    CNN_LABEL_STD  = cnn_stats['std'].astype(np.float32)    # (23,)

print(f"CNN label predictor loaded.  Label stats shape: "
      f"mean={CNN_LABEL_MEAN.shape}, std={CNN_LABEL_STD.shape}")


def cnn_predict_physical(flux_batch):
    """
    Run CNN on raw flux â†’ return (mean_physical, std_physical).
    Both are (batch, 23) in physical label units.

    The CNN's second output is a raw value; variance is recovered as
        var = softplus(raw) + 1e-4
    to match the training loss parameterisation.
    """
    flux_clean = flux_batch.copy()
    bad = ~np.isfinite(flux_clean) | (flux_clean <= 1e-3)
    flux_clean[bad] = 1.0

    mu_norm, raw_logvar = cnn_predictor(
        tf.constant(flux_clean, dtype=tf.float32), training=False
    )
    mu_phys = mu_norm.numpy() * CNN_LABEL_STD + CNN_LABEL_MEAN

    # Recover variance using the same softplus + floor as training
    var_norm = tf.nn.softplus(raw_logvar).numpy() + 1e-4
    std_phys = np.sqrt(var_norm) * CNN_LABEL_STD

    # Floor the uncertainty so the prior is never infinitely tight
    std_phys = np.maximum(std_phys, 0.01 * CNN_LABEL_STD)
    return mu_phys.astype(np.float32), std_phys.astype(np.float32)

Read data for imputation
274.00787
Imputing Labels for 149966 stars...
Converting [X/Fe] to [X/H]...
Transformation Complete. Final Input Shape: (149966, 27)
Read data for imputation
275.0918
(149966, 27) (27,) (27,)


I0000 00:00:1775154147.378094      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 186 variables whereas the saved optimizer has 190 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


CNN label predictor loaded.  Label stats shape: mean=(23,), std=(23,)


In [9]:
def select_stratified_test_sample(
    h5_path: str,
    stats_path: str,
    selected_labels: list,
    test_start: int,
    test_end: int,
    target_n: int = 5,
    logg_bins: int = 5,
    teff_bins: int = 10,
    mh_bins: int = 10,
) -> np.ndarray:
    """
    Selects a stratified test sample from the test window [test_start, test_end]
    by partitioning (LOGG, TEFF, M_H) space into a 3D histogram and picking
    the median-SNR star from each occupied bin.
    """

    max_bins = logg_bins * teff_bins * mh_bins
    if target_n > max_bins:
        raise ValueError(
            f"target_n ({target_n}) exceeds the number of bins "
            f"({logg_bins}x{teff_bins}x{mh_bins} = {max_bins}). "
            f"Reduce target_n or increase bin counts."
        )

    teff_idx = selected_labels.index('TEFF')
    logg_idx = selected_labels.index('LOGG')
    mh_idx   = selected_labels.index('M_H')

    print(f"Loading test window [{test_start}:{test_end}] "
          f"({test_end - test_start} stars)...")

    with h5py.File(h5_path, 'r') as f:
        meta  = f['metadata']
        teff  = meta['TEFF'][test_start:test_end].astype(np.float32)
        logg  = meta['LOGG'][test_start:test_end].astype(np.float32)
        m_h   = meta['M_H'][test_start:test_end].astype(np.float32)
        snr   = meta['SNR'][test_start:test_end].astype(np.float32)

    n_test = test_end - test_start

    SENTINEL = -5000.0
    valid = (teff > SENTINEL) & (logg > SENTINEL) & (m_h > SENTINEL)
    print(f"  Valid stars (no sentinel values): {valid.sum()} / {n_test}")

    def percentile_edges(values, mask, n_bins):
        pts = np.linspace(0, 100, n_bins + 1)
        edges = np.percentile(values[mask], pts)
        edges[0]  -= 1e-6
        edges[-1] += 1e-6
        return edges

    logg_edges = percentile_edges(logg, valid, logg_bins)
    teff_edges = percentile_edges(teff, valid, teff_bins)
    mh_edges   = percentile_edges(m_h,  valid, mh_bins)

    print(f"  LOGG edges: {np.round(logg_edges, 2)}")
    print(f"  TEFF edges: {np.round(teff_edges, 0)}")
    print(f"  M_H  edges: {np.round(mh_edges,  2)}")

    logg_bin = np.clip(np.digitize(logg, logg_edges) - 1, 0, logg_bins - 1)
    teff_bin = np.clip(np.digitize(teff, teff_edges) - 1, 0, teff_bins - 1)
    mh_bin   = np.clip(np.digitize(m_h,  mh_edges)  - 1, 0, mh_bins  - 1)

    local_indices_selected = []
    bin_summary = []

    for i in range(logg_bins):
        for j in range(teff_bins):
            for k in range(mh_bins):
                in_bin = valid & (logg_bin == i) & (teff_bin == j) & (mh_bin == k)
                bin_local_idx = np.where(in_bin)[0]

                if len(bin_local_idx) == 0:
                    continue

                bin_snr    = snr[bin_local_idx]
                median_snr = np.median(bin_snr)
                closest    = bin_local_idx[np.argmin(np.abs(bin_snr - median_snr))]

                local_indices_selected.append(closest)
                bin_summary.append({
                    'bin': (i, j, k),
                    'n_stars': len(bin_local_idx),
                    'median_snr': median_snr,
                    'selected_snr': snr[closest],
                })

    local_indices_selected = np.array(local_indices_selected)
    n_from_bins = len(local_indices_selected)
    print(f"\n  Occupied bins: {n_from_bins} / {max_bins}")

    if n_from_bins < target_n:
        shortfall = target_n - n_from_bins
        already_selected = set(local_indices_selected.tolist())
        remaining = np.array([
            idx for idx in np.where(valid)[0]
            if idx not in already_selected
        ])
        if len(remaining) < shortfall:
            print(f"  Warning: only {len(remaining)} additional valid stars "
                  f"available; test set will be smaller than target_n.")
            shortfall = len(remaining)

        rng = np.random.default_rng(42)
        extra = rng.choice(remaining, size=shortfall, replace=False)
        local_indices_selected = np.concatenate([local_indices_selected, extra])
        print(f"  Topped up with {shortfall} random valid stars "
              f"to reach {len(local_indices_selected)} total.")

    elif n_from_bins > target_n:
        occupancies = np.array([b['n_stars'] for b in bin_summary])
        order = np.argsort(occupancies)
        keep  = order[:target_n]
        local_indices_selected = local_indices_selected[keep]
        print(f"  Downsampled from {n_from_bins} to {target_n}, "
              f"prioritising sparse bins.")

    global_indices = np.sort(local_indices_selected + test_start)

    selected_teff = teff[global_indices - test_start]
    selected_logg = logg[global_indices - test_start]
    selected_mh   = m_h[global_indices  - test_start]
    selected_snr  = snr[global_indices  - test_start]

    print(f"\n{'='*50}")
    print(f"  Final test set: {len(global_indices)} stars")
    print(f"  TEFF  â€” min: {selected_teff.min():.0f}  "
               f"max: {selected_teff.max():.0f}  "
               f"median: {np.median(selected_teff):.0f}")
    print(f"  LOGG  â€” min: {selected_logg.min():.2f}  "
               f"max: {selected_logg.max():.2f}  "
               f"median: {np.median(selected_logg):.2f}")
    print(f"  [M/H] â€” min: {selected_mh.min():.2f}   "
               f"max: {selected_mh.max():.2f}   "
               f"median: {np.median(selected_mh):.2f}")
    print(f"  SNR   â€” min: {selected_snr.min():.1f}   "
               f"max: {selected_snr.max():.1f}   "
               f"median: {np.median(selected_snr):.1f}")
    print(f"{'='*50}\n")

    return global_indices


def load_test_spectra(h5_path, global_indices):
    """
    Loads flux and ivar for the selected test stars.
    """
    with h5py.File(h5_path, 'r') as f:
        flux = f['flux'][global_indices].astype(np.float32)
        ivar = f['ivar'][global_indices].astype(np.float32)
    return flux, ivar


# --- Run stratified selection ---
TEST_START = 140_000
TEST_END   = 149_500

global_indices = select_stratified_test_sample(
    h5_path        = config.H5_PATH,
    stats_path     = config.STATS_PATH,
    selected_labels= config.SELECTED_LABELS,
    test_start     = TEST_START,
    test_end       = TEST_END,
    target_n       = 5,
    logg_bins      = 5,
    teff_bins      = 10,
    mh_bins        = 10,
)

np.save("test_indices.npy", global_indices)
print(f"Saved {len(global_indices)} test indices to test_indices.npy")

flux, ivar = load_test_spectra(config.H5_PATH, global_indices)
print(f"Flux shape: {flux.shape}, IVAR shape: {ivar.shape}")

Loading test window [140000:149500] (9500 stars)...
  Valid stars (no sentinel values): 9457 / 9500
  LOGG edges: [0.   1.64 2.33 2.5  2.88 4.59]
  TEFF edges: [3203. 4007. 4265. 4474. 4613. 4702. 4781. 4861. 4959. 5413. 6725.]
  M_H  edges: [-2.25 -0.6  -0.46 -0.36 -0.27 -0.2  -0.12 -0.04  0.05  0.15  0.54]

  Occupied bins: 264 / 500
  Downsampled from 264 to 5, prioritising sparse bins.

  Final test set: 5 stars
  TEFF  â€” min: 4124  max: 4992  median: 4659
  LOGG  â€” min: 1.54  max: 3.27  median: 2.89
  [M/H] â€” min: -1.81   max: 0.35   median: -0.39
  SNR   â€” min: 123.2   max: 274.5   median: 220.1

Saved 5 test indices to test_indices.npy
Flux shape: (5, 8575), IVAR shape: (5, 8575)


In [10]:
import tensorflow_probability as tfp
tfb = tfp.bijectors

# Ordered exactly as SELECTED_LABELS
lower_bounds = [
    # Core
    3000.0, -0.5, -2.5,  0.0,  0.0,   0.0,  
    # CNO
    -1.5,  -1.0, -1.0, 
    # Metals
    -2.5,  -1.5, -1.0, -1.0, -1.5, -1.0, 
    -1.5,  -1.5, -1.5, -4.0, -2.5, -4.5, 
    -2.5,  -2.5
]

upper_bounds = [
    # Core
    20000.0, 5.0,  1.0,  3.0, 15.0, 80.0, 
    # CNO
    1.5,   3.5,  1.0,  
    # Metals
    1.0,   1.0,  1.5,  1.0,  1.0,  1.0,  
    1.5,   1.5,  1.0,  1.5,  2.0,  3.5,  
    1.5,   2.0
]

bounds_low = tf.constant(lower_bounds, dtype=tf.float32)
bounds_high = tf.constant(upper_bounds, dtype=tf.float32)

# Bijector to map Unconstrained space â†’ Bounded space (Physical Labels)
unconstrained_to_physical = tfb.Chain([
    tfb.Shift(shift=bounds_low),
    tfb.Scale(scale=(bounds_high - bounds_low)),
    tfb.Sigmoid()
])

# Inverse: Physical â†’ Unconstrained (for HMC initialisation)
physical_to_unconstrained = tfb.Invert(unconstrained_to_physical)



import json

# 1. Extract the raw architecture blueprint
config_m = model.get_config()
config_str = json.dumps(config_m)

# 2. Aggressively purge all low-precision flags
config_str = config_str.replace('"float16"', '"float32"')
config_str = config_str.replace('"mixed_float16"', '"float32"')

# 3. Rebuild the dictionary
clean_config = json.loads(config_str)

# 4. Construct a brand new graph using the sanitized config
model_pure_fp32 = model.__class__.from_config(clean_config)

# 5. Inject the trained weights
model_pure_fp32.set_weights(model.get_weights())

print("Model aggressively rebuilt in pure FP32.")


import time

# ============================================================
# CONSTANTS
# ============================================================
BATCH_SIZE_STARS = 10
TOTAL_STARS      = 5

N_LABELS_RAW = 23
N_LABELS_ALL = 27
N_PIXELS     = 8575

LABEL_NAMES = config.SELECTED_LABELS   # length 23, ordered

RESULTS_DIR = "/kaggle/working/mcmc_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

CHECKPOINT_PATH = os.path.join(RESULTS_DIR, "checkpoint.npz")
RESULTS_PATH    = os.path.join(RESULTS_DIR, "results.npz")

# â”€â”€ Stage 1: Core physics (9 params) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# TEFF, LOGG, M_H, VMICRO, VMACRO, VSINI, C_FE, N_FE, O_FE
CORE_INDICES  = [0, 1, 2, 3, 4, 5, 6, 7, 8]
N_CORE        = len(CORE_INDICES)                   # 9

NUM_CHAINS_CORE   = 8
NUM_RESULTS_CORE  = 500
NUM_BURNIN_CORE   = 400
NUM_ADAPT_CORE    = 320
MAX_TREE_DEPTH    = 6
TARGET_ACCEPT     = 0.8
PRIOR_WEIGHT      = 1.0

# â”€â”€ Stage 2: Individual abundances (14 elements, each 1D) â”€â”€â”€â”€
# FE_H, MG_FE, SI_FE, CA_FE, TI_FE, S_FE, AL_FE, MN_FE,
# NI_FE, CR_FE, K_FE, NA_FE, V_FE, CO_FE
ABUND_INDICES = [9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]
N_ABUND       = len(ABUND_INDICES)                  # 14

NUM_CHAINS_ELEM   = 4
NUM_RESULTS_ELEM  = 300
NUM_BURNIN_ELEM   = 200
NUM_ADAPT_ELEM    = 160

ABUND_LABEL_NAMES = [LABEL_NAMES[i] for i in ABUND_INDICES]

# â”€â”€ Element pixel masks from apogee_mask.npy â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
apogee_mask_path = "/kaggle/input/datasets/aneeshshastri/element-masks/apogee_mask.npy"
_raw_mask = np.load(apogee_mask_path)   # (8575, 27) â€” 0 = sensitive

ELEMENT_PIXEL_MASKS = {}   # label_index â†’ (8575,) float32 tensor
for abund_idx in ABUND_INDICES:
    col = abund_idx
    mask_col = _raw_mask[:, col]
    # Convention: 0.0 = line present / sensitive.
    sensitive = (mask_col > 0.01).astype(np.float32)
    n_pix = int(sensitive.sum())
    ELEMENT_PIXEL_MASKS[abund_idx] = tf.constant(sensitive, tf.float32)
    print(f"  {LABEL_NAMES[abund_idx]:>6s}: {n_pix:>5d} sensitive pixels")

for idx in ABUND_INDICES:
    if tf.reduce_sum(ELEMENT_PIXEL_MASKS[idx]).numpy() < 10:
        print(f"  âš  {LABEL_NAMES[idx]} has <10 pixels, using full spectrum")
        ELEMENT_PIXEL_MASKS[idx] = tf.ones([N_PIXELS], tf.float32)

Model aggressively rebuilt in pure FP32.
    FE_H:  1933 sensitive pixels
   MG_FE:   105 sensitive pixels
   SI_FE:    96 sensitive pixels
   CA_FE:    19 sensitive pixels
   TI_FE:    31 sensitive pixels
    S_FE:     8 sensitive pixels
   AL_FE:    47 sensitive pixels
   MN_FE:    63 sensitive pixels
   NI_FE:   181 sensitive pixels
   CR_FE:    42 sensitive pixels
    K_FE:    14 sensitive pixels
   NA_FE:    13 sensitive pixels
    V_FE:    27 sensitive pixels
   CO_FE:    23 sensitive pixels
  âš  S_FE has <10 pixels, using full spectrum


In [11]:
def get_27_features(labels_23):
    """
    labels_23 : (..., 23)  â€” any leading batch dimensions
    returns    : (..., 27)
    """
    teff   = labels_23[ :, config.SELECTED_LABELS.index('TEFF')]
    logg   = labels_23[ :, config.SELECTED_LABELS.index('LOGG')]
    vmacro = labels_23[ :, config.SELECTED_LABELS.index('VMACRO')]
    vsini  = labels_23[ :, config.SELECTED_LABELS.index('VSINI')]
    c_fe   = labels_23[ :, config.SELECTED_LABELS.index('C_FE')]
    o_fe   = labels_23[ :, config.SELECTED_LABELS.index('O_FE')]
    m_h    = labels_23[ :, config.SELECTED_LABELS.index('M_H')]

    inv_teff  = 5040.0 / (teff + 1e-6)
    v_broad   = tf.sqrt(tf.square(vmacro) + tf.square(vsini))
    c_minus_o = c_fe - o_fe
    log_pe    = 0.5 * logg + 0.5 * m_h

    eng = tf.stack([inv_teff, v_broad, c_minus_o, log_pe], axis=-1)
    return tf.concat([labels_23, eng], axis=-1)

In [12]:
_cnn_mu_var = tf.Variable(
    tf.zeros([BATCH_SIZE_STARS, N_LABELS_RAW], tf.float32), trainable=False
)
_cnn_std_var = tf.Variable(
    tf.ones([BATCH_SIZE_STARS, N_LABELS_RAW], tf.float32), trainable=False
)
_fixed_abund_var = tf.Variable(
    tf.zeros([BATCH_SIZE_STARS, N_ABUND], tf.float32), trainable=False
)


@tf.function(reduce_retracing=True)
def _assemble_full_23(core_params, fixed_abund):
    """Splice core (B*C, 9) and abundance (B*C, 14) into (B*C, 23)."""
    parts = []
    for i in range(N_LABELS_RAW):
        if i in CORE_INDICES:
            ci = CORE_INDICES.index(i)
            parts.append(core_params[:, ci:ci+1])
        else:
            ai = ABUND_INDICES.index(i)
            parts.append(fixed_abund[:, ai:ai+1])
    return tf.concat(parts, axis=1)


@tf.function(reduce_retracing=True)
def core_log_prob_fn(theta_core, obs_flux, obs_ivar):
    """
    theta_core : (C, B, 9)   â€” core physics, physical space
    returns    : (C, B)
    Full-spectrum Ï‡Â² + CNN prior on core params.
    """
    num_chains  = tf.shape(theta_core)[0]
    batch_stars = tf.shape(theta_core)[1]

    theta_BCL  = tf.transpose(theta_core, [1, 0, 2])
    theta_flat = tf.reshape(theta_BCL, [-1, N_CORE])
    abund_rep  = tf.repeat(_fixed_abund_var, num_chains, axis=0)
    full_23    = _assemble_full_23(theta_flat, abund_rep)

    labels_27   = get_27_features(full_23)
    labels_norm = (labels_27 - MEAN_TENSOR) / STD_TENSOR

    model_flux = model_pure_fp32(labels_norm, training=False)
    model_flux = tf.cast(model_flux, tf.float32)  # guard against any residual fp16 in model output

    if len(model_flux.shape) == 3:
        model_flux = model_flux[:, :, 0]

    obs_flux_rep = tf.repeat(obs_flux, num_chains, axis=0)
    obs_ivar_rep = tf.repeat(obs_ivar, num_chains, axis=0)

    safe_flux = tf.where(tf.math.is_finite(obs_flux_rep), obs_flux_rep, 0.0)
    safe_ivar = tf.where(
        tf.math.is_finite(obs_ivar_rep) & tf.math.is_finite(obs_flux_rep),
        obs_ivar_rep, 0.0
    )
    safe_ivar = 1.0 / (1.0 / (safe_ivar) + 1e-5)

    flux_valid = tf.cast((safe_flux > 1e-3) & (safe_flux < 1.3), tf.float32)
    ivar_valid = tf.cast(safe_ivar > 0.0, tf.float32)
    mask       = flux_valid * ivar_valid

    chi2_flat = tf.reduce_sum(
        tf.square(safe_flux - model_flux) * safe_ivar * mask, axis=1
    )
    log_lik_flat = -0.5 * chi2_flat

    cnn_mu_core  = tf.gather(_cnn_mu_var,  CORE_INDICES, axis=1)
    cnn_std_core = tf.gather(_cnn_std_var, CORE_INDICES, axis=1)
    cnn_mu_rep   = tf.repeat(cnn_mu_core,  num_chains, axis=0)
    cnn_std_rep  = tf.repeat(cnn_std_core, num_chains, axis=0)
    log_prior = -0.5 * tf.reduce_sum(
        tf.square((theta_flat - cnn_mu_rep) / cnn_std_rep), axis=1
    )

    log_post = log_lik_flat + PRIOR_WEIGHT * log_prior
    log_post_BC = tf.reshape(log_post, [batch_stars, num_chains])
    return tf.transpose(log_post_BC)

In [13]:
_fixed_full_var = tf.Variable(
    tf.zeros([BATCH_SIZE_STARS, N_LABELS_RAW], tf.float32), trainable=False
)
_elem_col_var = tf.Variable(0, dtype=tf.int32, trainable=False)
_elem_pixel_mask_var = tf.Variable(
    tf.ones([N_PIXELS], tf.float32), trainable=False
)


@tf.function(reduce_retracing=True)
def element_log_prob_fn(theta_1d, obs_flux, obs_ivar):
    """
    theta_1d : (C, B, 1)   â€” single abundance, physical space
    returns  : (C, B)
    Element-masked Ï‡Â² + CNN prior on the single element.
    """
    num_chains  = tf.shape(theta_1d)[0]
    batch_stars = tf.shape(theta_1d)[1]
    elem_col    = _elem_col_var

    theta_BCL  = tf.transpose(theta_1d, [1, 0, 2])
    theta_flat = tf.reshape(theta_BCL, [-1, 1])

    full_rep = tf.repeat(_fixed_full_var, num_chains, axis=0)
    bc = tf.shape(full_rep)[0]
    row_idx = tf.range(bc)
    col_idx = tf.fill([bc], elem_col)
    indices = tf.stack([row_idx, col_idx], axis=1)
    full_spliced = tf.tensor_scatter_nd_update(
        full_rep, indices, theta_flat[:, 0]
    )

    labels_27   = get_27_features(full_spliced)
    labels_norm = (labels_27 - MEAN_TENSOR) / STD_TENSOR

    model_flux = model_pure_fp32(labels_norm, training=False)
    model_flux = tf.cast(model_flux, tf.float32)  # guard against any residual fp16 in model output
    if len(model_flux.shape) == 3:
        model_flux = model_flux[:, :, 0]

    obs_flux_rep = tf.repeat(obs_flux, num_chains, axis=0)
    obs_ivar_rep = tf.repeat(obs_ivar, num_chains, axis=0)

    safe_flux = tf.where(tf.math.is_finite(obs_flux_rep), obs_flux_rep, 0.0)
    safe_ivar = tf.where(
        tf.math.is_finite(obs_ivar_rep) & tf.math.is_finite(obs_flux_rep),
        obs_ivar_rep, 0.0
    )
    safe_ivar = 1.0 / (1.0 / (safe_ivar) + 1e-5)

    flux_valid = tf.cast((safe_flux > 1e-3) & (safe_flux < 1.3), tf.float32)
    ivar_valid = tf.cast(safe_ivar > 0.0, tf.float32)
    elem_mask  = tf.reshape(_elem_pixel_mask_var, [1, N_PIXELS])
    mask       = flux_valid * ivar_valid * elem_mask

    chi2_flat = tf.reduce_sum(
        tf.square(safe_flux - model_flux) * safe_ivar * mask, axis=1
    )
    log_lik_flat = -0.5 * chi2_flat

    cnn_mu_elem  = _cnn_mu_var[:, elem_col:elem_col+1]
    cnn_std_elem = _cnn_std_var[:, elem_col:elem_col+1]
    cnn_mu_rep   = tf.repeat(cnn_mu_elem,  num_chains, axis=0)
    cnn_std_rep  = tf.repeat(cnn_std_elem, num_chains, axis=0)
    log_prior = -0.5 * tf.reduce_sum(
        tf.square((theta_flat - cnn_mu_rep) / cnn_std_rep), axis=1
    )

    log_post = log_lik_flat + PRIOR_WEIGHT * log_prior
    log_post_BC = tf.reshape(log_post, [batch_stars, num_chains])
    return tf.transpose(log_post_BC)

In [14]:
def make_init_state_core(batch_stars, num_chains, physical_init_23):
    """Returns (C, B, 9) in physical space â€” core subset."""
    margin     = 1e-3
    safe_lower = bounds_low  + margin
    safe_upper = bounds_high - margin

    core_init = tf.gather(
        tf.cast(physical_init_23, tf.float32), CORE_INDICES, axis=1
    )
    base = tf.tile(tf.expand_dims(core_init, 0), [num_chains, 1, 1])
    core_std = tf.gather(STD_TENSOR[:N_LABELS_RAW], CORE_INDICES)
    jitter = tf.random.normal(tf.shape(base)) * (0.05 * core_std)
    core_low  = tf.gather(safe_lower, CORE_INDICES)
    core_high = tf.gather(safe_upper, CORE_INDICES)
    return tf.clip_by_value(base + jitter, core_low, core_high)


def make_init_state_elem(batch_stars, num_chains, phys_value_1d):
    """Returns (C, B, 1) in physical space."""
    base = tf.reshape(tf.cast(phys_value_1d, tf.float32), [1, -1, 1])
    base = tf.tile(base, [num_chains, 1, 1])
    jitter = tf.random.normal(tf.shape(base)) * 0.02
    return base + jitter

In [15]:
_core_low_tf  = tf.gather(bounds_low,  CORE_INDICES)
_core_high_tf = tf.gather(bounds_high, CORE_INDICES)
_core_bijector = tfb.Chain([
    tfb.Shift(_core_low_tf),
    tfb.Scale(_core_high_tf - _core_low_tf),
    tfb.Sigmoid(),
])
_core_inv_bijector = tfb.Invert(_core_bijector)

_obs_flux_var = tf.Variable(
    tf.zeros([BATCH_SIZE_STARS, N_PIXELS], tf.float32), trainable=False
)
_obs_ivar_var = tf.Variable(
    tf.zeros([BATCH_SIZE_STARS, N_PIXELS], tf.float32), trainable=False
)
_init_core_var = tf.Variable(
    tf.zeros([NUM_CHAINS_CORE, BATCH_SIZE_STARS, N_CORE], tf.float32),
    trainable=False,
)


def sample_core_compiled():
    """NUTS for Stage 1: 9D core physics, full spectrum."""
    def log_prob_closure(theta_unc):
        return core_log_prob_fn(
            _core_bijector.forward(theta_unc),
            _obs_flux_var, _obs_ivar_var
        )

    step_size = tf.fill([1, BATCH_SIZE_STARS, N_CORE], 0.01)
    nuts = tfp.mcmc.NoUTurnSampler(
        target_log_prob_fn=log_prob_closure,
        step_size=step_size,
        max_tree_depth=MAX_TREE_DEPTH,
    )
    adaptive = tfp.mcmc.DualAveragingStepSizeAdaptation(
        inner_kernel=nuts,
        num_adaptation_steps=NUM_ADAPT_CORE,
        target_accept_prob=TARGET_ACCEPT,
        step_size_setter_fn=lambda pkr, s: pkr._replace(step_size=s),
        step_size_getter_fn=lambda pkr: pkr.step_size,
        log_accept_prob_getter_fn=lambda pkr: pkr.log_accept_ratio,
    )
    init_unc = _core_inv_bijector.forward(_init_core_var)
    return tfp.mcmc.sample_chain(
        num_results=NUM_RESULTS_CORE,
        num_burnin_steps=NUM_BURNIN_CORE,
        current_state=init_unc,
        kernel=adaptive,
        trace_fn=lambda _, pkr: pkr.inner_results.is_accepted,
    )

In [16]:
_elem_low_var  = tf.Variable(-3.0, dtype=tf.float32, trainable=False)
_elem_high_var = tf.Variable(3.0,  dtype=tf.float32, trainable=False)

_init_elem_var = tf.Variable(
    tf.zeros([NUM_CHAINS_ELEM, BATCH_SIZE_STARS, 1], tf.float32),
    trainable=False,
)


def sample_element_compiled():
    """NUTS for Stage 2: 1D abundance, element-masked spectrum."""
    lo = tf.reshape(_elem_low_var, [1])
    hi = tf.reshape(_elem_high_var, [1])
    elem_bij = tfb.Chain([
        tfb.Shift(lo), tfb.Scale(hi - lo), tfb.Sigmoid(),
    ])
    elem_inv = tfb.Invert(elem_bij)

    def log_prob_closure(theta_unc):
        return element_log_prob_fn(
            elem_bij.forward(theta_unc),
            _obs_flux_var, _obs_ivar_var
        )

    step_size = tf.fill([1, BATCH_SIZE_STARS, 1], 0.01)
    nuts = tfp.mcmc.NoUTurnSampler(
        target_log_prob_fn=log_prob_closure,
        step_size=step_size,
        max_tree_depth=MAX_TREE_DEPTH,
    )
    adaptive = tfp.mcmc.DualAveragingStepSizeAdaptation(
        inner_kernel=nuts,
        num_adaptation_steps=NUM_ADAPT_ELEM,
        target_accept_prob=TARGET_ACCEPT,
        step_size_setter_fn=lambda pkr, s: pkr._replace(step_size=s),
        step_size_getter_fn=lambda pkr: pkr.step_size,
        log_accept_prob_getter_fn=lambda pkr: pkr.log_accept_ratio,
    )
    init_unc = elem_inv.forward(_init_elem_var)
    return tfp.mcmc.sample_chain(
        num_results=NUM_RESULTS_ELEM,
        num_burnin_steps=NUM_BURNIN_ELEM,
        current_state=init_unc,
        kernel=adaptive,
        trace_fn=lambda _, pkr: pkr.inner_results.is_accepted,
    )

In [17]:
def run_core_mcmc(obs_flux, obs_ivar, physical_init_23,
                  cnn_mean, cnn_std):
    """Returns core_medians (B,9), lo (B,9), hi (B,9), acc, rhat (9,)."""
    obs_flux = tf.cast(obs_flux, tf.float32)
    obs_ivar = tf.cast(obs_ivar, tf.float32)

    _cnn_mu_var.assign(tf.cast(cnn_mean, tf.float32))
    _cnn_std_var.assign(tf.cast(cnn_std, tf.float32))

    abund_init = tf.gather(
        tf.cast(cnn_mean, tf.float32), ABUND_INDICES, axis=1
    )
    _fixed_abund_var.assign(abund_init)

    init_core = make_init_state_core(
        BATCH_SIZE_STARS, NUM_CHAINS_CORE, physical_init_23
    )

    _obs_flux_var.assign(obs_flux)
    _obs_ivar_var.assign(obs_ivar)
    _init_core_var.assign(init_core)

    samples_unc, is_accepted = sample_core_compiled()
    samples_phys = _core_bijector.forward(samples_unc)

    r_hat = tf.reduce_mean(
        tfp.mcmc.diagnostic.potential_scale_reduction(
            samples_phys, independent_chain_ndims=1
        ), axis=0
    ).numpy()

    n_total = NUM_RESULTS_CORE * NUM_CHAINS_CORE
    flat = tf.reshape(samples_phys, [n_total, -1, N_CORE])
    pcts = tfp.stats.percentile(flat, [16.0, 50.0, 84.0], axis=0)

    medians = pcts[1].numpy()
    lo_sig  = (pcts[1] - pcts[0]).numpy()
    hi_sig  = (pcts[2] - pcts[1]).numpy()
    accept  = tf.reduce_mean(tf.cast(is_accepted, tf.float32)).numpy()

    return medians, lo_sig, hi_sig, accept, r_hat

In [18]:
def run_element_mcmc(obs_flux, obs_ivar, fixed_labels_23, elem_idx,
                     cnn_mean, cnn_std):
    """Returns elem_median (B,), lo (B,), hi (B,), acc, rhat."""
    obs_flux = tf.cast(obs_flux, tf.float32)
    obs_ivar = tf.cast(obs_ivar, tf.float32)

    _fixed_full_var.assign(tf.cast(fixed_labels_23, tf.float32))
    _elem_col_var.assign(elem_idx)
    _elem_pixel_mask_var.assign(ELEMENT_PIXEL_MASKS[elem_idx])
    _elem_low_var.assign(bounds_low[elem_idx])
    _elem_high_var.assign(bounds_high[elem_idx])

    init_val = cnn_mean[:, elem_idx]
    init_state = make_init_state_elem(
        BATCH_SIZE_STARS, NUM_CHAINS_ELEM, init_val
    )

    _obs_flux_var.assign(obs_flux)
    _obs_ivar_var.assign(obs_ivar)
    _init_elem_var.assign(init_state)

    samples_unc, is_accepted = sample_element_compiled()

    lo_val = bounds_low[elem_idx]
    hi_val = bounds_high[elem_idx]
    bij = tfb.Chain([
        tfb.Shift(tf.constant([lo_val])),
        tfb.Scale(tf.constant([hi_val - lo_val])),
        tfb.Sigmoid(),
    ])
    samples_phys = bij.forward(samples_unc)

    r_hat = tf.reduce_mean(
        tfp.mcmc.diagnostic.potential_scale_reduction(
            samples_phys, independent_chain_ndims=1
        )
    ).numpy()

    n_total = NUM_RESULTS_ELEM * NUM_CHAINS_ELEM
    flat = tf.reshape(samples_phys, [n_total, -1, 1])
    pcts = tfp.stats.percentile(flat, [16.0, 50.0, 84.0], axis=0)

    median = pcts[1, :, 0].numpy()
    lo_sig = (pcts[1, :, 0] - pcts[0, :, 0]).numpy()
    hi_sig = (pcts[2, :, 0] - pcts[1, :, 0]).numpy()
    accept = tf.reduce_mean(tf.cast(is_accepted, tf.float32)).numpy()

    return median, lo_sig, hi_sig, accept, r_hat

In [19]:
def load_checkpoint():
    fresh = {
        'global_indices':    [],
        'true_labels':       [],
        'inferred_labels':   [],
        'lower_sigma':       [],
        'upper_sigma':       [],
        'aspcap_errors':     [],
        'acceptance_rates':  [],
        'r_hat':             [],
        'wall_seconds':      [],
    }
    if os.path.exists(CHECKPOINT_PATH):
        data = np.load(CHECKPOINT_PATH, allow_pickle=True)
        results = {k: list(data[k]) for k in fresh}
        n_done  = len(results['global_indices'])
        print(f"  Resuming from checkpoint: {n_done} stars already done.")
        return results, n_done
    return fresh, 0


def save_checkpoint(results):
    np.savez(CHECKPOINT_PATH, **{
        k: np.array(v) for k, v in results.items()
    })


def save_final_results(results):
    arrays = {k: np.array(v) for k, v in results.items()}
    arrays['label_names'] = np.array(LABEL_NAMES)
    np.savez(RESULTS_PATH, **arrays)
    print(f"\nFinal results saved to {RESULTS_PATH}")
    print(f"  Keys: {list(arrays.keys())}")

In [20]:
def run_inference_pipeline(
    test_indices,
    true_labels_norm,
    aspcap_errors,
    flux_array,
    ivar_array,
):
    """
    Two-stage FERRE-style inference:
      Stage 1 â€” NUTS on 9 core physics params (full spectrum)
      Stage 2 â€” NUTS on each of 14 abundances individually (masked pixels)
    """
    n_stars = len(test_indices)
    assert n_stars <= len(flux_array)

    true_labels_physical = (
        true_labels_norm[:, :N_LABELS_RAW] * STD_TENSOR[:N_LABELS_RAW]
        + MEAN_TENSOR[:N_LABELS_RAW]
    )

    results, n_done = load_checkpoint()
    remaining_local = list(range(n_done, n_stars))

    print(f"\n{'='*65}")
    print(f"  Two-Stage Inference: {n_stars} stars, batch {BATCH_SIZE_STARS}")
    print(f"  Stage 1 (core {N_CORE}D): {NUM_CHAINS_CORE}ch, "
          f"{NUM_BURNIN_CORE} burn, {NUM_RESULTS_CORE} samp")
    print(f"  Stage 2 ({N_ABUND} elem Ã— 1D): {NUM_CHAINS_ELEM}ch, "
          f"{NUM_BURNIN_ELEM} burn, {NUM_RESULTS_ELEM} samp")
    print(f"  Prior weight: {PRIOR_WEIGHT}")
    print(f"  To do: {len(remaining_local)} stars")
    print(f"{'='*65}\n")

    total_start = time.time()
    batch_count = 0
    CHECKPOINT_EVERY = 5

    for batch_start in range(0, len(remaining_local), BATCH_SIZE_STARS):
        batch_local = remaining_local[batch_start:batch_start + BATCH_SIZE_STARS]
        actual_batch = len(batch_local)

        batch_flux = flux_array[batch_local]
        batch_ivar = ivar_array[batch_local]

        # CNN warm-start
        cnn_mu, cnn_sig = cnn_predict_physical(batch_flux)

        # Pad if needed
        if actual_batch < BATCH_SIZE_STARS:
            pad_n = BATCH_SIZE_STARS - actual_batch
            batch_flux = np.concatenate([
                batch_flux, np.zeros((pad_n, N_PIXELS), np.float32)])
            batch_ivar = np.concatenate([
                batch_ivar, np.zeros((pad_n, N_PIXELS), np.float32)])
            cnn_mu  = np.concatenate([
                cnn_mu,  np.tile(cnn_mu[:1],  (pad_n, 1))])
            cnn_sig = np.concatenate([
                cnn_sig, np.tile(cnn_sig[:1], (pad_n, 1))])

        # â•â•â•â•â•â•â•â•â•â•â• STAGE 1: Core (9D, full spectrum) â•â•â•â•â•â•â•â•â•â•â•
        t_s1 = time.time()
        core_med, core_lo, core_hi, s1_acc, s1_rhat = run_core_mcmc(
            obs_flux=batch_flux, obs_ivar=batch_ivar,
            physical_init_23=cnn_mu,
            cnn_mean=cnn_mu, cnn_std=cnn_sig,
        )
        t_s1_elapsed = time.time() - t_s1
        s1_tag = "âœ“" if s1_rhat.max() < 1.05 else f"âš {s1_rhat.max():.3f}"
        print(f"    S1 core: acc={s1_acc:.2f}  RÌ‚ {s1_tag}  {t_s1_elapsed:.1f}s")

        # Build fixed labels: core from Stage 1 + abund from CNN
        fixed_23 = cnn_mu.copy()
        for ci, gi in enumerate(CORE_INDICES):
            fixed_23[:, gi] = core_med[:, ci]

        # â•â•â•â•â•â•â•â•â•â•â• STAGE 2: Abundances (1D Ã— 14) â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
        t_s2 = time.time()
        abund_med = np.zeros((BATCH_SIZE_STARS, N_ABUND), np.float32)
        abund_lo  = np.zeros((BATCH_SIZE_STARS, N_ABUND), np.float32)
        abund_hi  = np.zeros((BATCH_SIZE_STARS, N_ABUND), np.float32)
        s2_rhats, s2_accs = [], []

        for ai, elem_idx in enumerate(ABUND_INDICES):
            e_med, e_lo, e_hi, e_acc, e_rhat = run_element_mcmc(
                obs_flux=batch_flux, obs_ivar=batch_ivar,
                fixed_labels_23=fixed_23,
                elem_idx=elem_idx,
                cnn_mean=cnn_mu, cnn_std=cnn_sig,
            )
            abund_med[:, ai] = e_med
            abund_lo[:, ai]  = e_lo
            abund_hi[:, ai]  = e_hi
            s2_rhats.append(e_rhat)
            s2_accs.append(e_acc)
            # Feed forward: update fixed_23 with this element's result
            fixed_23[:, elem_idx] = e_med

        t_s2_elapsed = time.time() - t_s2
        s2_tag = "âœ“" if max(s2_rhats) < 1.05 else f"âš {max(s2_rhats):.3f}"
        print(f"    S2 elem: acc={np.mean(s2_accs):.2f}  RÌ‚ {s2_tag}  "
              f"{t_s2_elapsed:.1f}s  ({N_ABUND} elements)")

        # â•â•â•â•â•â•â•â•â•â•â• ASSEMBLE RESULTS â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
        full_med = np.zeros((BATCH_SIZE_STARS, N_LABELS_RAW), np.float32)
        full_lo  = np.zeros((BATCH_SIZE_STARS, N_LABELS_RAW), np.float32)
        full_hi  = np.zeros((BATCH_SIZE_STARS, N_LABELS_RAW), np.float32)
        full_rhat = np.zeros(N_LABELS_RAW, np.float32)

        for ci, gi in enumerate(CORE_INDICES):
            full_med[:, gi] = core_med[:, ci]
            full_lo[:, gi]  = core_lo[:, ci]
            full_hi[:, gi]  = core_hi[:, ci]
            full_rhat[gi]   = s1_rhat[ci]

        for ai, gi in enumerate(ABUND_INDICES):
            full_med[:, gi] = abund_med[:, ai]
            full_lo[:, gi]  = abund_lo[:, ai]
            full_hi[:, gi]  = abund_hi[:, ai]
            full_rhat[gi]   = s2_rhats[ai]

        elapsed = t_s1_elapsed + t_s2_elapsed
        avg_acc = (s1_acc + np.mean(s2_accs)) / 2

        for b in range(actual_batch):
            local_idx  = batch_local[b]
            global_idx = int(test_indices[local_idx])

            results['global_indices'].append(global_idx)
            results['true_labels'].append(true_labels_physical[local_idx])
            results['inferred_labels'].append(full_med[b])
            results['lower_sigma'].append(full_lo[b])
            results['upper_sigma'].append(full_hi[b])
            results['aspcap_errors'].append(
                aspcap_errors[local_idx, :N_LABELS_RAW])
            results['acceptance_rates'].append(avg_acc)
            results['r_hat'].append(full_rhat)
            results['wall_seconds'].append(elapsed / actual_batch)

        n_done += actual_batch
        batch_count += 1

        if batch_count % CHECKPOINT_EVERY == 0 or n_done >= n_stars:
            save_checkpoint(results)

        est_left = (n_stars - n_done) * elapsed / actual_batch / 60
        print(f"  [{n_done:>4}/{n_stars}]  batch {batch_count}  |  "
              f"{elapsed:.1f}s  (~{est_left:.0f} min left)\n")

    total_elapsed = time.time() - total_start
    print(f"\nAll {n_stars} stars done in {total_elapsed/60:.1f} min.")

    save_final_results(results)
    return results

In [21]:
def load_results(path=RESULTS_PATH):
    """
    Loads the saved results and returns a clean dict with named arrays.

    Example usage:
        r = load_results()
        teff_inferred = r['inferred_labels'][:, 0]
        teff_true     = r['true_labels'][:, 0]
        teff_err      = r['aspcap_errors'][:, 0]
    """
    data = np.load(path, allow_pickle=True)
    r = {k: data[k] for k in data.files}

    if 'label_names' in r:
        r['label_index'] = {
            name: i for i, name in enumerate(r['label_names'])
        }

    print("Loaded results:")
    print(f"  Stars:  {len(r['global_indices'])}")
    print(f"  Labels: {r.get('label_names', ['(not stored)'])}")
    print(f"  Keys:   {[k for k in r if k != 'label_names']}")
    return r

In [ ]:
test_indices = np.load("/kaggle/working/test_indices.npy")
results = run_inference_pipeline(
    test_indices      = test_indices,
    true_labels_norm  = labels,
    aspcap_errors     = label_err,
    flux_array        = real_data,
    ivar_array        = real_data_ivar,
)


  Two-Stage Inference: 5 stars, batch 10
  Stage 1 (core 9D): 8ch, 400 burn, 500 samp
  Stage 2 (14 elem Ã— 1D): 4ch, 200 burn, 300 samp
  Prior weight: 1.0
  To do: 5 stars



I0000 00:00:1775154175.204322      55 cuda_dnn.cc:529] Loaded cuDNN version 91002
